
# Smart MCQ Solver — Qwen 7B Knowledge Model

The previous notebook scored about **0.73** because TF-IDF and answer-position models learned dataset patterns but could not reliably answer unseen general-knowledge questions.

This notebook uses:

- **Qwen2.5-7B-Instruct** in 4-bit mode
- Three option-order permutations to reduce A/B/C/D/E position bias
- Next-token probabilities to produce a proper top-3 ranking
- Only high-confidence exact/prompt duplicate transfer from training data
- MAP@3 validation on prompt-grouped unseen questions

> Kaggle notebook settings: enable a GPU and Internet access for the first model download.


In [ ]:

# Install only the packages needed by this notebook.
!pip install -q -U "transformers>=4.45.0" accelerate bitsandbytes sentencepiece


In [ ]:

import os
import re
import gc
import math
import random
import warnings
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from sklearn.model_selection import GroupShuffleSplit
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

OPTIONS = list("ABCDE")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


## 1. Load train, test, and sample submission

In [ ]:

def find_competition_file(filename):
    matches = []
    for root, _, files in os.walk("/kaggle/input"):
        if filename in files:
            matches.append(os.path.join(root, filename))

    if not matches:
        raise FileNotFoundError(
            f"{filename} was not found. Add the competition dataset to this notebook."
        )

    # Prefer a path containing all three standard competition files.
    return matches[0]

TRAIN_PATH = find_competition_file("train.csv")
TEST_PATH = find_competition_file("test.csv")
SAMPLE_PATH = find_competition_file("sample_submission.csv")

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_PATH)

required_train = {"id", "prompt", "A", "B", "C", "D", "E", "answer"}
required_test = {"id", "prompt", "A", "B", "C", "D", "E"}

assert required_train.issubset(train_df.columns)
assert required_test.issubset(test_df.columns)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
display(train_df.head(2))


## 2. MAP@3 and duplicate-safe helper functions

In [ ]:

def normalize_text(value):
    value = str(value).lower().strip()
    value = re.sub(r"\s+", " ", value)
    value = re.sub(r"[^a-z0-9 ]+", "", value)
    return value.strip()

def prompt_key(row):
    return normalize_text(row["prompt"])

def full_signature(row):
    return tuple(
        [normalize_text(row["prompt"])] +
        [normalize_text(row[o]) for o in OPTIONS]
    )

def map_at_3(actual, ranked_predictions):
    scores = []
    for answer, ranking in zip(actual, ranked_predictions):
        top3 = ranking[:3]
        if answer in top3:
            scores.append(1.0 / (top3.index(answer) + 1))
        else:
            scores.append(0.0)
    return float(np.mean(scores))

def build_safe_answer_lookups(df):
    exact_lookup = {}
    prompt_correct_texts = defaultdict(list)

    for _, row in df.iterrows():
        exact_lookup[full_signature(row)] = row["answer"]
        correct_text = normalize_text(row[row["answer"]])
        prompt_correct_texts[prompt_key(row)].append(correct_text)

    # Keep prompt-level transfer only when every training occurrence agrees
    # on the same correct answer text.
    safe_prompt_lookup = {}
    for key, texts in prompt_correct_texts.items():
        unique_texts = set(texts)
        if len(unique_texts) == 1:
            safe_prompt_lookup[key] = next(iter(unique_texts))

    return exact_lookup, safe_prompt_lookup

def apply_safe_duplicate_override(df, rankings, exact_lookup, prompt_lookup):
    updated = [list(r) for r in rankings]
    exact_count = 0
    prompt_count = 0

    for i, (_, row) in enumerate(df.iterrows()):
        selected = None

        signature = full_signature(row)
        if signature in exact_lookup:
            selected = exact_lookup[signature]
            exact_count += 1
        else:
            key = prompt_key(row)
            known_answer_text = prompt_lookup.get(key)

            if known_answer_text is not None:
                matching_options = [
                    o for o in OPTIONS
                    if normalize_text(row[o]) == known_answer_text
                ]
                if len(matching_options) == 1:
                    selected = matching_options[0]
                    prompt_count += 1

        if selected is not None:
            updated[i] = [selected] + [o for o in updated[i] if o != selected]

    return updated, exact_count, prompt_count


## 3. Load Qwen2.5-7B-Instruct in 4-bit mode

In [ ]:

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

if DEVICE != "cuda":
    raise RuntimeError(
        "A GPU is required. In Kaggle, open Notebook settings and enable a GPU accelerator."
    )

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
    trust_remote_code=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

model.eval()
print("Loaded:", MODEL_NAME)



## 4. Position-bias-resistant MCQ scoring

For every question, the options are shown in three different orders.  
The model scores the next answer letter, and each score is mapped back to the original option.


In [ ]:

PERMUTATIONS = [
    [0, 1, 2, 3, 4],  # original
    [4, 3, 2, 1, 0],  # reversed
    [2, 3, 4, 0, 1],  # rotated
]

SYSTEM_MESSAGE = (
    "You are an expert multiple-choice question solver. "
    "Use factual knowledge and careful reasoning. "
    "Return only the single capital letter of the best answer."
)

def build_user_prompt(row, permutation):
    displayed_options = []
    for displayed_index, original_index in enumerate(permutation):
        displayed_letter = OPTIONS[displayed_index]
        original_letter = OPTIONS[original_index]
        displayed_options.append(
            f"{displayed_letter}. {str(row[original_letter]).strip()}"
        )

    return (
        f"Question: {str(row['prompt']).strip()}\n\n"
        + "\n".join(displayed_options)
        + "\n\nChoose the best answer. Reply with exactly one letter: A, B, C, D, or E."
    )

def candidate_first_token_ids(tokenizer):
    ids = {}
    for letter in OPTIONS:
        variants = [letter, " " + letter, "\n" + letter]
        token_ids = set()

        for text in variants:
            encoded = tokenizer.encode(text, add_special_tokens=False)
            if encoded:
                token_ids.add(encoded[0])

        ids[letter] = sorted(token_ids)

    return ids

LETTER_TOKEN_IDS = candidate_first_token_ids(tokenizer)
print("Candidate token IDs:", LETTER_TOKEN_IDS)

@torch.inference_mode()
def score_prompt_batch(user_prompts, batch_size=4, max_length=1800):
    all_scores = []

    for start in range(0, len(user_prompts), batch_size):
        batch_prompts = user_prompts[start:start + batch_size]

        chats = [
            [
                {"role": "system", "content": SYSTEM_MESSAGE},
                {"role": "user", "content": prompt},
            ]
            for prompt in batch_prompts
        ]

        rendered = [
            tokenizer.apply_chat_template(
                chat,
                tokenize=False,
                add_generation_prompt=True,
            )
            for chat in chats
        ]

        encoded = tokenizer(
            rendered,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length,
        ).to(model.device)

        outputs = model(**encoded)
        last_positions = encoded["attention_mask"].sum(dim=1) - 1
        batch_indices = torch.arange(len(rendered), device=model.device)
        next_logits = outputs.logits[batch_indices, last_positions, :].float()

        for row_logits in next_logits:
            letter_logits = []
            for letter in OPTIONS:
                variant_logits = row_logits[LETTER_TOKEN_IDS[letter]]
                # Log-sum-exp combines valid tokenization variants.
                letter_logits.append(torch.logsumexp(variant_logits, dim=0))

            letter_logits = torch.stack(letter_logits)
            probabilities = torch.softmax(letter_logits, dim=0)
            all_scores.append(probabilities.detach().cpu().numpy())

        del encoded, outputs, next_logits
        torch.cuda.empty_cache()

    return np.asarray(all_scores)

def qwen_mcq_probabilities(df, batch_size=4):
    accumulated = np.zeros((len(df), 5), dtype=np.float64)

    for permutation_number, permutation in enumerate(PERMUTATIONS, start=1):
        prompts = [
            build_user_prompt(row, permutation)
            for _, row in df.iterrows()
        ]

        displayed_probabilities = score_prompt_batch(
            prompts,
            batch_size=batch_size,
        )

        # displayed index -> original index
        for displayed_index, original_index in enumerate(permutation):
            accumulated[:, original_index] += displayed_probabilities[:, displayed_index]

        print(f"Finished permutation {permutation_number}/{len(PERMUTATIONS)}")

    accumulated /= len(PERMUTATIONS)
    accumulated /= accumulated.sum(axis=1, keepdims=True)
    return accumulated

def probabilities_to_rankings(probabilities):
    return [
        [OPTIONS[j] for j in np.argsort(-row)]
        for row in probabilities
    ]


## 5. Honest validation on unseen prompt groups

In [ ]:

# A limited validation sample keeps notebook runtime manageable.
# Identical normalized prompts are kept entirely in train or validation.

validation_groups = train_df["prompt"].map(normalize_text)
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.12,
    random_state=SEED,
)

_, validation_indices = next(
    splitter.split(train_df, groups=validation_groups)
)

validation_df = train_df.iloc[validation_indices].reset_index(drop=True)

# Cap validation to 240 rows, while keeping a deterministic sample.
if len(validation_df) > 240:
    validation_df = validation_df.sample(240, random_state=SEED).reset_index(drop=True)

print("Validation rows:", len(validation_df))

validation_probabilities = qwen_mcq_probabilities(
    validation_df,
    batch_size=4,
)
validation_rankings = probabilities_to_rankings(validation_probabilities)

validation_score = map_at_3(
    validation_df["answer"].tolist(),
    validation_rankings,
)
validation_top1 = np.mean([
    ranking[0] == answer
    for ranking, answer in zip(
        validation_rankings,
        validation_df["answer"].tolist(),
    )
])

print(f"Qwen validation MAP@3: {validation_score:.6f}")
print(f"Qwen validation top-1 accuracy: {validation_top1:.6f}")


## 6. Score the test set and apply only safe train/test transfers

In [ ]:

test_probabilities = qwen_mcq_probabilities(
    test_df,
    batch_size=4,
)
test_rankings = probabilities_to_rankings(test_probabilities)

exact_lookup, safe_prompt_lookup = build_safe_answer_lookups(train_df)

final_rankings, exact_overrides, prompt_overrides = apply_safe_duplicate_override(
    test_df,
    test_rankings,
    exact_lookup,
    safe_prompt_lookup,
)

print("Exact full-question overrides:", exact_overrides)
print("Safe prompt/answer-text overrides:", prompt_overrides)


## 7. Create `submission.csv`

In [ ]:

submission = sample_submission.copy()
submission["Prediction"] = [
    " ".join(ranking[:3])
    for ranking in final_rankings
]

assert len(submission) == len(test_df)
assert submission["Prediction"].notna().all()
assert submission["Prediction"].str.split().map(len).eq(3).all()
assert submission["Prediction"].apply(
    lambda value: len(set(value.split())) == 3
).all()
assert submission["Prediction"].apply(
    lambda value: set(value.split()).issubset(set(OPTIONS))
).all()

submission.to_csv("submission.csv", index=False)

display(submission.head(10))
print("Saved submission.csv")



## Why this should outperform the 0.73 notebook

The previous solution relied heavily on shallow text patterns and answer-position correlations.  
This version uses a 7-billion-parameter instruction model that can answer the factual questions directly.

The final public score is only known after submitting `submission.csv`; no notebook can guarantee a specific hidden-label score. The validation cell reports the model's measured MAP@3 before submission.
